Copyright **`(c)`** 2022 Giovanni Squillero `<squillero@polito.it>`  
[`https://github.com/squillero/computational-intelligence`](https://github.com/squillero/computational-intelligence)  
Free for personal or classroom use; see [`LICENSE.md`](https://github.com/squillero/computational-intelligence/blob/master/LICENSE.md) for details.  


# Lab 3: Policy Search

## Task

Write agents able to play [*Nim*](https://en.wikipedia.org/wiki/Nim), with an arbitrary number of rows and an upper bound $k$ on the number of objects that can be removed in a turn (a.k.a., *subtraction game*).

The player **taking the last object wins**.

* Task3.1: An agent using fixed rules based on *nim-sum* (i.e., an *expert system*)
* Task3.2: An agent using evolved rules
* Task3.3: An agent using minmax
* Task3.4: An agent using reinforcement learning

## Instructions

* Create the directory `lab3` inside the course repo 
* Put a `README.md` and your solution (all the files, code and auxiliary data if needed)

## Notes

* Working in group is not only allowed, but recommended (see: [Ubuntu](https://en.wikipedia.org/wiki/Ubuntu_philosophy) and [Cooperative Learning](https://files.eric.ed.gov/fulltext/EJ1096789.pdf)). Collaborations must be explicitly declared in the `README.md`.
* [Yanking](https://www.emacswiki.org/emacs/KillingAndYanking) from the internet is allowed, but sources must be explicitly declared in the `README.md`.

## Deadlines ([AoE](https://en.wikipedia.org/wiki/Anywhere_on_Earth))

* Sunday, December 4th for Task3.1 and Task3.2
* Sunday, December 11th for Task3.3 and Task3.4
* Sunday, December 18th for all reviews

In [201]:
import logging
from collections import namedtuple
import random
from typing import Callable
from copy import deepcopy
from itertools import accumulate
from operator import xor


## The *Nim* and *Nimply* classes

In [202]:
Nimply = namedtuple("Nimply", "row, num_objects")


In [203]:
class Nim:
    def __init__(self, num_rows: int, k: int = None) -> None:
        self._rows = [i * 2 + 1 for i in range(num_rows)]
        self._k = k

    def __bool__(self):
        return sum(self._rows) > 0

    def __str__(self):
        return "<" + " ".join(str(_) for _ in self._rows) + ">"

    @property
    def rows(self) -> tuple:
        return tuple(self._rows)

    @property
    def k(self) -> int:
        return self._k

    def nimming(self, ply: Nimply) -> None:
        row, num_objects = ply
        assert self._rows[row] >= num_objects
        assert self._k is None or num_objects <= self._k
        self._rows[row] -= num_objects


In [204]:
def active_rows(state: Nim) -> int:
    return sum(o > 0 for o in state.rows)


# Task 3.1

## Fixed rules

In [205]:
def pure_random(state: Nim) -> Nimply:
    row = random.choice([r for r, c in enumerate(state.rows) if c > 0])
    num_objects = random.randint(1, state.rows[row])
    return Nimply(row, num_objects)


In [206]:
NUM_MATCHES = 200
NIM_SIZE = 19


def evaluate(strategy: Callable) -> float:
    opponent = (strategy, pure_random)
    won = 0

    for m in range(NUM_MATCHES):
        nim = Nim(NIM_SIZE)
        player = 0
        while nim:
            ply = opponent[player](nim)
            nim.nimming(ply)
            player = 1 - player
        if player == 1:
            won += 1
    return won / NUM_MATCHES


In [207]:
def pick_maximum_from_highest_row(state: Nim) -> Nimply:
    """Pick always the maximum possible number of the highest row"""
    possible_moves = [(r, o) for r, c in enumerate(state.rows)
                      for o in range(1, c + 1)]
    return Nimply(*max(possible_moves, key=lambda m: (m[0], m[1])))


def pick_minimum_from_lowest_row(state: Nim) -> Nimply:
    """Pick always the minimum possible number of the lowest row"""
    possible_moves = [(r, o) for r, c in enumerate(state.rows)
                      for o in range(1, c + 1)]
    return Nimply(*max(possible_moves, key=lambda m: (-m[0], -m[1])))


def pick_maximum_from_lowest_row(state: Nim) -> Nimply:
    """Pick always the maximum possible number of the lowest row"""
    possible_moves = [(r, o) for r, c in enumerate(state.rows)
                      for o in range(1, c + 1)]
    return Nimply(*max(possible_moves, key=lambda m: (-m[0], m[1])))


def pick_minimum_from_highest_row(state: Nim) -> Nimply:
    """Pick always the minimum possible number of the highest row"""
    possible_moves = [(r, o) for r, c in enumerate(state.rows)
                      for o in range(1, c + 1)]
    return Nimply(*max(possible_moves, key=lambda m: (m[0], -m[1])))


In [208]:
evaluate(pick_maximum_from_highest_row)


0.69

In [209]:
evaluate(pick_minimum_from_lowest_row)


0.42

In [210]:
evaluate(pick_maximum_from_lowest_row)


0.765

In [211]:
evaluate(pick_minimum_from_highest_row)


0.395

In [212]:
def count_and_decide(state: Nim) -> Nimply:
    if active_rows(state) % 2 == 0:
        return pick_maximum_from_highest_row(state)
    else:
        return pick_minimum_from_lowest_row(state)


evaluate(count_and_decide)


0.345

In [213]:
def count_and_decide(state: Nim) -> Nimply:
    if active_rows(state) % 2 != 0:
        return pick_maximum_from_highest_row(state)
    else:
        return pick_minimum_from_lowest_row(state)


evaluate(count_and_decide)


0.855

In [214]:
def count_and_decide(state: Nim) -> Nimply:
    if active_rows(state) % 2 == 0:
        return pick_maximum_from_highest_row(state)
    else:
        return pick_maximum_from_lowest_row(state)


evaluate(count_and_decide)


0.735

In [215]:
def count_and_decide(state: Nim) -> Nimply:
    if active_rows(state) % 2 != 0:
        return pick_maximum_from_highest_row(state)
    else:
        return pick_maximum_from_lowest_row(state)


evaluate(count_and_decide)


0.79

In [216]:
def count_and_decide(state: Nim) -> Nimply:
    if active_rows(state) % 2 == 0:
        return pick_maximum_from_highest_row(state)
    else:
        return pick_minimum_from_highest_row(state)


evaluate(count_and_decide)


0.265

In [217]:
def count_and_decide(state: Nim) -> Nimply:
    if active_rows(state) % 2 != 0:
        return pick_maximum_from_highest_row(state)
    else:
        return pick_minimum_from_highest_row(state)


evaluate(count_and_decide)


0.84

In [218]:
def count_and_decide(state: Nim) -> Nimply:
    if active_rows(state) % 2 == 0:
        return pick_minimum_from_lowest_row(state)
    else:
        return pick_maximum_from_lowest_row(state)


evaluate(count_and_decide)


0.91

In [219]:
def count_and_decide(state: Nim) -> Nimply:
    if active_rows(state) % 2 != 0:
        return pick_minimum_from_lowest_row(state)
    else:
        return pick_maximum_from_lowest_row(state)


evaluate(count_and_decide)


0.275

In [220]:
def count_and_decide(state: Nim) -> Nimply:
    if active_rows(state) % 2 == 0:
        return pick_minimum_from_lowest_row(state)
    else:
        return pick_minimum_from_highest_row(state)


evaluate(count_and_decide)


0.41

In [221]:
def count_and_decide(state: Nim) -> Nimply:
    if active_rows(state) % 2 != 0:
        return pick_minimum_from_lowest_row(state)
    else:
        return pick_minimum_from_highest_row(state)


evaluate(count_and_decide)


0.385

In [222]:
def count_and_decide(state: Nim) -> Nimply:
    if active_rows(state) % 2 == 0:
        return pick_maximum_from_lowest_row(state)
    else:
        return pick_minimum_from_highest_row(state)


evaluate(count_and_decide)


0.34

In [223]:
def count_and_decide(state: Nim) -> Nimply:
    if active_rows(state) % 2 != 0:
        return pick_maximum_from_lowest_row(state)
    else:
        return pick_minimum_from_highest_row(state)


evaluate(count_and_decide)


0.84

In [224]:
def pick_all_elements(state: Nim) -> Nimply:
    row = random.choice([r for r, c in enumerate(state.rows) if c > 0])
    return Nimply(row, state.rows[row])


def pick_all_but_one_elements(state: Nim) -> Nimply:
    row = random.choice([r for r, c in enumerate(state.rows) if c > 0])
    return Nimply(row, state.rows[row] - 1)


In [225]:
def count_and_decide(state: Nim) -> Nimply:
    if active_rows(state) % 2 == 0:
        return pick_all_but_one_elements(state)
    else:
        return pick_all_elements(state)


evaluate(count_and_decide)


1.0

## Task 3.2

In [250]:
def make_strategy(genome: str) -> Callable:
    def evolvable(state: Nim) -> Nimply:

        if random.random() < float(genome):
            ply = pick_maximum_from_highest_row(state)
        else:
            ply = pick_maximum_from_lowest_row(state)
        return ply

    return evolvable


In [249]:
Individual = namedtuple("Individual", ["genome", "fitness"])
POPULATION_SIZE = 50
NUM_GENERATIONS = 50
OFFSPRING_SIZE = 10


In [245]:
def compute_fitness(genome, strategy):
    return evaluate(strategy(genome))


def tournament(population, tournament_size=2):
    return max(random.choices(population, k=tournament_size), key=lambda i: i.fitness)


def mutation():
    return str(random.random())


def crossover(g_1, g_2):
    g_x = (float(g_1) + float(g_2))/2
    return str(g_x)


In [266]:
def my_genetic_algorithm(population, strategy):
    for generation in range(NUM_GENERATIONS):
        offspring = list()
        for i in range(OFFSPRING_SIZE):
            if random.random() < 0.5:
                p = tournament(population)
                o = mutation()
            else:
                # promising genome 1
                p1 = tournament(population)
                # promising genome 2
                p2 = tournament(population)
                o = crossover(p1.genome, p2.genome)
            f = compute_fitness(o, strategy)
            offspring.append(Individual(o, f))

        population += offspring
        population = sorted(population, key=lambda i: i.fitness, reverse=True)[
            :POPULATION_SIZE]

        best_so_far = population[0]
        if (generation % 5 == 0):
            print(f"#{generation}\t\t\tGENOME (Probability): {best_so_far.genome}\tFITNESS: {best_so_far.fitness}")

In [267]:
def evolution(evolvable_strategy):
    population = list()
    for _ in range(POPULATION_SIZE):
        genome = str(random.random())
        population.append(Individual(genome, compute_fitness(genome, evolvable_strategy)))

    my_genetic_algorithm(population, evolvable_strategy)

In [268]:
evolution(make_strategy)

#0			GENOME (Probability): 0.013911602435984327	FITNESS: 0.865
#5			GENOME (Probability): 0.013911602435984327	FITNESS: 0.865
#10			GENOME (Probability): 0.17638901060016793	FITNESS: 0.875
#15			GENOME (Probability): 0.09678799411862435	FITNESS: 0.88
#20			GENOME (Probability): 0.09678799411862435	FITNESS: 0.88
#25			GENOME (Probability): 0.09678799411862435	FITNESS: 0.88
#30			GENOME (Probability): 0.09678799411862435	FITNESS: 0.88
#35			GENOME (Probability): 0.09678799411862435	FITNESS: 0.88
#40			GENOME (Probability): 0.09678799411862435	FITNESS: 0.88
#45			GENOME (Probability): 0.09678799411862435	FITNESS: 0.88
